In [3]:
!pip install "protobuf>=3.20,<6" "rich>=10.14.0,<14"

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.



  Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl.metadata (592 bytes)
Using cached protobuf-5.29.6-cp310-abi3-win_amd64.whl (435 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.1
    Uninstalling protobuf-7.35.1:
      Successfully uninstalled protobuf-7.35.1
  Attempting uninstall: rich
    Found existing installation: rich 15.0.0
    Uninstalling rich-15.0.0:
      Successfully uninstalled rich-15.0.0


In [7]:
import cv2
import time
from ultralytics import YOLO

def run_tracker():
    print("[*] Loading YOLOv8 model...")
    model = YOLO('yolov8n.pt')

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("[-] Cannot access camera")
        return

    # 🎥 Video writer (save output)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter('output.avi', fourcc, 20.0, (640, 480))

    prev_time = 0
    object_ids = set()

    print("[+] Press 'Q' to exit")

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Resize for consistency
            frame = cv2.resize(frame, (640, 480))

            # YOLO Tracking
            results = model.track(frame, persist=True, verbose=False)

            annotated_frame = results[0].plot()

            # 🎯 COUNT OBJECTS
            if results[0].boxes.id is not None:
                ids = results[0].boxes.id.cpu().numpy()
                for i in ids:
                    object_ids.add(int(i))

            total_objects = len(object_ids)

            # 📊 FPS CALCULATION
            current_time = time.time()
            fps = 1 / (current_time - prev_time) if prev_time != 0 else 0
            prev_time = current_time

            # 🖥️ OVERLAY TEXT
            cv2.putText(annotated_frame, f"FPS: {int(fps)}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.putText(annotated_frame, f"Objects: {total_objects}", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

            # 🎥 Save video
            out.write(annotated_frame)

            # Show window
            cv2.imshow("YOLOv8 Advanced Tracking", annotated_frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    finally:
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        print("[+] Saved as output.avi")
        print("[+] Clean exit")

if __name__ == "__main__":
    run_tracker()

[*] Loading YOLOv8 model...
[+] Press 'Q' to exit
[+] Saved as output.avi
[+] Clean exit
